# 08 — Unified Pipeline v2 + SecBERT Comparison

**Objective:** wrap everything we built into one function, `analyze_report_v2()`, and demonstrate swapping a **domain-pretrained** model (SecBERT) for vanilla BERT to see whether the domain-matched weights help.

## What this notebook is

The wrap-up. CTI 201 has produced two trained heads that work together:

- **NER** (from notebook 04, trained on DNRTI): given a report, extract threat entities like HackOrg, Tool, SamFile, Area, ...
- **Tactic classifier** (from notebook 07, trained on TRAM): given a report, predict which MITRE ATT&CK tactics it describes.

This notebook glues them together so a caller can hand in a raw report and get back a structured dict of entities + tactics — the output format an analyst or a downstream system would consume.

It also runs a **domain-model experiment**: retrain the tactic classifier using `jackaduma/SecBERT` (BERT pretrained on cybersecurity corpora) instead of `bert-base-cased`, and measure whether the F1 changes. This is the classic "does domain pretraining help?" comparison.

## What this notebook does

1. Load the DNRTI NER model and the TRAM tactic classifier (with tuned per-tactic thresholds)
2. Build `analyze_report_v2(text)` — one call, structured output
3. Demo it on a hand-written example and on a real APTnotes report
4. Fine-tune SecBERT on TRAM tactics with identical hyperparameters
5. Compare BERT vs SecBERT on the same test set — a direct A/B
6. (Optional) update `analyze_report_v2()` to use whichever classifier won

## Prerequisites

- Notebook 04: `models/ner_dnrti_final/`
- Notebook 07: `models/tactic_clf_final/` (with `thresholds.json`)
- Notebook 03 (optional, for the APTnotes demo): `processed/aptnotes_pool/`
- Notebook 02: `processed/tram_tactics/` (for the SecBERT retrain)

## Step 1 — Load both heads

Each head is its own BERT model with its own tokenizer. We wrap them in a tiny class so `analyze_report_v2()` can call them cleanly.

In [17]:
import json
from pathlib import Path
from dataclasses import dataclass, field
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification

NER_DIR = Path('models/ner_dnrti_final')
CLF_DIR = Path('models/tactic_clf_final')

assert NER_DIR.exists(), 'Run notebook 04 first.'
assert CLF_DIR.exists(), 'Run notebook 07 first.'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- NER ---
ner_tok = AutoTokenizer.from_pretrained(NER_DIR)
ner_model = AutoModelForTokenClassification.from_pretrained(NER_DIR).to(device).eval()
with open(NER_DIR / 'tag_vocab.json') as f:
    ner_tags = json.load(f)['tag_list']
id2tag = {i: t for i, t in enumerate(ner_tags)}

# --- Tactic classifier ---
clf_tok = AutoTokenizer.from_pretrained(CLF_DIR)
clf_model = AutoModelForSequenceClassification.from_pretrained(CLF_DIR).to(device).eval()
with open(CLF_DIR / 'tactic_vocab.json') as f:
    tactic_vocab = json.load(f)
tactics = tactic_vocab['tactics']
num_tactics = len(tactics)

# Tuned thresholds from notebook 07; fall back to 0.5 if missing.
thresh_path = CLF_DIR / 'thresholds.json'
if thresh_path.exists():
    with open(thresh_path) as f:
        thr_cfg = json.load(f)
    tactic_thresholds = np.array(
        [thr_cfg['per_tactic'].get(t, 0.5) for t in tactics], dtype=np.float32,
    )
else:
    tactic_thresholds = np.full(num_tactics, 0.5, dtype=np.float32)

print(f'NER model: {len(ner_tags)} tag classes, device={device}')
print(f'Tactic classifier: {num_tactics} tactics')
print(f'Using per-tactic thresholds: {thresh_path.exists()}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

NER model: 27 tag classes, device=cuda
Tactic classifier: 14 tactics
Using per-tactic thresholds: True


## Step 2 — Helpers: extract entity spans, extract tactics

These mirror the logic from notebooks 05 and 07, now packaged as reusable functions rather than inline cells.

In [18]:
NER_MAX_LEN = 256
CLF_MAX_LEN = 512
CLF_STRIDE = 128


def _bio_spans(words: list[str], tags: list[str]) -> list[dict]:
    """Walk BIO tags and return span dicts."""
    spans, start, etype = [], None, None
    for i, t in enumerate(tags):
        if t.startswith('B-'):
            if start is not None:
                spans.append({'type': etype, 'text': ' '.join(words[start:i])})
            start, etype = i, t[2:]
        elif t.startswith('I-') and start is not None and t[2:] == etype:
            continue
        else:
            if start is not None:
                spans.append({'type': etype, 'text': ' '.join(words[start:i])})
                start, etype = None, None
    if start is not None:
        spans.append({'type': etype, 'text': ' '.join(words[start:])})
    return spans


def extract_entities(text: str) -> list[dict]:
    words = text.split()
    if not words:
        return []

    enc = ner_tok(words, is_split_into_words=True, truncation=True,
                  max_length=NER_MAX_LEN, return_tensors='pt').to(device)
    word_ids = ner_tok(words, is_split_into_words=True, truncation=True,
                      max_length=NER_MAX_LEN).word_ids()
    with torch.no_grad():
        pred_ids = ner_model(**enc).logits[0].argmax(-1).cpu().tolist()

    word_tags = ['O'] * len(words)
    seen = set()
    for wid, pid in zip(word_ids, pred_ids):
        if wid is None or wid in seen or wid >= len(word_tags):
            continue
        seen.add(wid)
        word_tags[wid] = id2tag[pid]

    return _bio_spans(words, word_tags)

In [19]:
def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def extract_tactics(text: str) -> list[dict]:
    """Return list of {tactic, prob, threshold} for tactics that crossed their threshold."""
    if not text.strip():
        return []

    enc = clf_tok(text, truncation=True, max_length=CLF_MAX_LEN, stride=CLF_STRIDE,
                  return_overflowing_tokens=True, padding=True, return_tensors='pt')
    enc.pop('overflow_to_sample_mapping', None)
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = clf_model(input_ids=enc['input_ids'],
                           attention_mask=enc['attention_mask']).logits
    probs = _sigmoid(logits.cpu().numpy()).max(axis=0)   # max-pool chunks per tactic

    out = []
    for i, p in enumerate(probs):
        if p >= tactic_thresholds[i]:
            out.append({
                'tactic': tactics[i],
                'prob': float(p),
                'threshold': float(tactic_thresholds[i]),
            })
    out.sort(key=lambda r: -r['prob'])
    return out

## Step 3 — `analyze_report_v2()`

One call. Takes a raw report string (or text extracted from a PDF), returns a structured dict.

In [20]:
def analyze_report_v2(text: str) -> dict:
    """Unified NER + tactic analysis for a single CTI report."""
    entities = extract_entities(text)

    # Group entities by type for easier consumption.
    by_type: dict[str, list[str]] = {}
    for e in entities:
        by_type.setdefault(e['type'], []).append(e['text'])
    # Deduplicate within each type while preserving order.
    for t, items in by_type.items():
        seen = set()
        uniq = []
        for it in items:
            if it not in seen:
                seen.add(it)
                uniq.append(it)
        by_type[t] = uniq

    return {
        'num_words': len(text.split()),
        'entities': by_type,
        'tactics': extract_tactics(text),
    }

## Step 4 — Demo: a hand-written example

Short synthetic report. Good for sanity-checking both heads at once.

In [21]:
sample = (
    "APT29 deployed Cobalt Strike against financial institutions in Europe in 2023. "
    "The attackers gained initial access via spearphishing attachments, then escalated "
    "privileges using stolen credentials and moved laterally through the corporate network. "
    "Persistence was maintained with a scheduled task. Sensitive customer data was "
    "exfiltrated over DNS tunneling."
)

result = analyze_report_v2(sample)

print(f'Report length: {result["num_words"]} words')
print('\nEntities (by type):')
for etype, items in sorted(result['entities'].items()):
    print(f'  {etype:10s}  {items}')

print('\nTactics (above per-tactic threshold):')
if not result['tactics']:
    print('  (none)')
for t in result['tactics']:
    print(f'  {t["tactic"]:24s}  prob={t["prob"]:.3f}  (threshold={t["threshold"]:.2f})')

Report length: 47 words

Entities (by type):
  Area        ['Europe']
  HackOrg     ['APT29']
  Org         ['financial institutions']
  Tool        ['Cobalt']
  Way         ['spearphishing', 'DNS']

Tactics (above per-tactic threshold):
  privilege-escalation      prob=0.956  (threshold=0.57)
  persistence               prob=0.912  (threshold=0.20)
  initial-access            prob=0.832  (threshold=0.17)
  defense-evasion           prob=0.760  (threshold=0.30)


## Step 5 — Demo on a real APTnotes report

Pull an actual extracted PDF from the notebook 03 pool and run the pipeline end-to-end.

In [22]:
from datasets import load_from_disk

APT_POOL = Path('processed/aptnotes_pool')
if not APT_POOL.exists():
    print('APTnotes pool not available — skipping real-world demo.')
else:
    pool = load_from_disk(str(APT_POOL))
    example = next((r for r in pool if len(r['text'].split()) > 500), pool[0])
    print(f'Report:  {example["title"][:90]}')
    print(f'Source:  {example["source"]}')
    print(f'Length:  ~{len(example["text"].split())} words')

    # Cap to a reasonable length so the demo doesn't take forever on CPU.
    text_for_demo = example['text'][:6000]
    res = analyze_report_v2(text_for_demo)

    print(f'\nEntities extracted:')
    for etype, items in sorted(res['entities'].items()):
        items_str = ', '.join(items[:6])
        more = f'  (+{len(items) - 6} more)' if len(items) > 6 else ''
        print(f'  {etype:10s}  {items_str}{more}')

    print(f'\nPredicted tactics:')
    if not res['tactics']:
        print('  (none)')
    for t in res['tactics']:
        print(f'  {t["tactic"]:24s}  prob={t["prob"]:.3f}')

Report:  Crude Faux: An Analysis Of Cyber Conflict Within The Oil & Gas Industries
Source:  CERIAS
Length:  ~18412 words

Entities extracted:
  HackOrg     mul-
  Idus        oil & gas industries, Oil & Gas, oil & gas industry
  SecTeam     Kambic,
  Time        2013-9

Predicted tactics:
  command-and-control       prob=0.505
  initial-access            prob=0.314


## Step 6 — Does a domain model help? Fine-tune SecBERT on TRAM tactics

`jackaduma/SecBERT` is a BERT-base model pretrained on cybersecurity text (APTnotes, Stucco-Data, CASIE — APT reports, CTI, security events). Same architecture as `bert-base-cased`, different weights — it's seen a lot more cyber vocabulary during pretraining. We picked it over the original `ehsanaghaei/SecureBERT` because that repo has broken HF metadata (no safetensors, disabled discussions) which causes load errors in recent `transformers` versions.

We'll fine-tune it on the **exact same** TRAM data with the **exact same** hyperparameters we used in notebook 07. Any F1 difference is attributable to the pretraining.

This is a real retrain — budget ~10–15 min on CPU, a couple of minutes on GPU.

In [23]:
from datasets import load_from_disk
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, hamming_loss, f1_score


SECBERT_CKPT = 'jackaduma/SecBERT'
TRAM_DIR = Path('processed/tram_tactics')
assert TRAM_DIR.exists(), 'Run notebook 02 first.'

tram = load_from_disk(str(TRAM_DIR))

sec_tok = AutoTokenizer.from_pretrained(SECBERT_CKPT)
sec_model = AutoModelForSequenceClassification.from_pretrained(
    SECBERT_CKPT,
    num_labels=num_tactics,
    problem_type='multi_label_classification',
    use_safetensors=True,
)


def tokenize_sec(batch):
    return sec_tok(batch['text'], truncation=True, max_length=128)


tok_sec = tram.map(tokenize_sec, batched=True,
                   remove_columns=['text', 'tactic_names'])
print(tok_sec)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: jackaduma/SecBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3256
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 407
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 407
    })
})


In [24]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = (_sigmoid(logits) >= 0.5).astype(np.float32)
    return {
        'subset_accuracy': accuracy_score(labels, preds),
        'hamming_accuracy': 1.0 - hamming_loss(labels, preds),
        'micro_f1': f1_score(labels, preds, average='micro', zero_division=0),
        'macro_f1': f1_score(labels, preds, average='macro', zero_division=0),
    }


sec_args = TrainingArguments(
    output_dir='models/tactic_secbert',
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    report_to='none',
)

sec_trainer = Trainer(
    model=sec_model,
    args=sec_args,
    train_dataset=tok_sec['train'],
    eval_dataset=tok_sec['validation'],
    processing_class=sec_tok,
    data_collator=DataCollatorWithPadding(sec_tok),
    compute_metrics=compute_metrics,
)

sec_trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Subset Accuracy,Hamming Accuracy,Micro F1,Macro F1
1,0.264837,0.249103,0.194103,0.907862,0.331210,0.065673
2,0.179895,0.173051,0.442260,0.935942,0.632427,0.357917
3,0.136241,0.147781,0.550369,0.948578,0.726424,0.507312
4,0.098745,0.138208,0.601966,0.952264,0.751825,0.526751
5,0.076207,0.136514,0.611794,0.954019,0.763538,0.534596
6,0.066239,0.131269,0.611794,0.953843,0.763276,0.536877
7,0.057287,0.132376,0.614251,0.954370,0.768271,0.539008
8,0.048166,0.131918,0.614251,0.954370,0.767857,0.552432


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1632, training_loss=0.1361381690583977, metrics={'train_runtime': 472.7732, 'train_samples_per_second': 55.096, 'train_steps_per_second': 3.452, 'total_flos': 630729441967872.0, 'train_loss': 0.1361381690583977, 'epoch': 8.0})

## Step 7 — Head-to-head: BERT vs SecBERT on TRAM test

Evaluate SecBERT with a plain 0.5 threshold (no per-tactic tuning, to keep the comparison apples-to-apples). Compare against the BERT tactic classifier from notebook 07 evaluated the same way.

In [25]:
# SecBERT test metrics
sec_test = sec_trainer.evaluate(tok_sec['test'])

# Recompute BERT test metrics with default 0.5 threshold, for fairness
bert_clf = AutoModelForSequenceClassification.from_pretrained(CLF_DIR).to(device).eval()
bert_tok = AutoTokenizer.from_pretrained(CLF_DIR)

tok_bert = tram.map(
    lambda b: bert_tok(b['text'], truncation=True, max_length=128),
    batched=True,
    remove_columns=['text', 'tactic_names'],
)

bert_eval = Trainer(
    model=bert_clf,
    args=TrainingArguments(output_dir='models/_tmp_bert_eval',
                           per_device_eval_batch_size=32,
                           report_to='none'),
    eval_dataset=tok_bert['test'],
    processing_class=bert_tok,
    data_collator=DataCollatorWithPadding(bert_tok),
    compute_metrics=compute_metrics,
).evaluate()

print(f'{"metric":18s}  {"BERT":>10s}  {"SecBERT":>10s}  diff')
print('-' * 52)
for k in ['subset_accuracy', 'hamming_accuracy', 'micro_f1', 'macro_f1']:
    b = bert_eval[f'eval_{k}']
    s = sec_test[f'eval_{k}']
    print(f'{k:18s}  {b:10.4f}  {s:10.4f}  {s - b:+.4f}')

Training Loss,Validation Loss,Epoch,Subset Accuracy,Hamming Accuracy,Micro F1,Macro F1
0.048166,0.132197,8,0.616708,0.954545,0.774194,0.542589


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/3256 [00:00<?, ? examples/s]

Map:   0%|          | 0/407 [00:00<?, ? examples/s]

Map:   0%|          | 0/407 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Subset Accuracy,Hamming Accuracy,Micro F1,Macro F1
No log,0.138614,0,0.631450,0.955598,0.775510,0.523105


metric                    BERT     SecBERT  diff
----------------------------------------------------
subset_accuracy         0.6314      0.6167  -0.0147
hamming_accuracy        0.9556      0.9545  -0.0011
micro_f1                0.7755      0.7742  -0.0013
macro_f1                0.5231      0.5426  +0.0195


### How to read this

- **SecBERT wins clearly on macro F1** → domain pretraining helped, especially on rare/technical tactics. Worth swapping in.
- **Tie or SecBERT-marginal** → pretraining helps some but doesn't move the needle much on this small of a fine-tuning set. Vanilla BERT is a reasonable default.
- **SecBERT loses** → possible if the tokenizer mismatch hurts, or if SecBERT's pretraining corpus doesn't match TRAM's vocabulary as closely as advertised. Try lowering the learning rate (SecBERT sometimes prefers 1e-5).

Note this comparison uses the **plain 0.5 threshold** for both. If SecBERT wins at 0.5, tuning thresholds (as in notebook 07 Step 6) could widen its lead further — threshold tuning tends to lift whichever model is underlying.

## Step 8 — Save SecBERT, optionally swap it into the pipeline

In [26]:
SEC_FINAL = Path('models/tactic_secbert_final')
SEC_FINAL.mkdir(parents=True, exist_ok=True)

sec_trainer.save_model(str(SEC_FINAL))
sec_tok.save_pretrained(str(SEC_FINAL))
with open(SEC_FINAL / 'tactic_vocab.json', 'w') as f:
    json.dump({'tactics': tactics, 'tactic2id': {t: i for i, t in enumerate(tactics)}},
              f, indent=2)

print(f'Saved SecBERT to {SEC_FINAL}')
print('\nTo use SecBERT in analyze_report_v2(), reload clf_model/clf_tok from')
print(f'{SEC_FINAL} and rerun the pipeline demo. (Thresholds were BERT-tuned; either')
print('rerun threshold tuning for SecBERT or use the default 0.5 for a fair swap.)')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved SecBERT to models/tactic_secbert_final

To use SecBERT in analyze_report_v2(), reload clf_model/clf_tok from
models/tactic_secbert_final and rerun the pipeline demo. (Thresholds were BERT-tuned; either
rerun threshold tuning for SecBERT or use the default 0.5 for a fair swap.)


## Summary

| | What |
|---|---|
| Unified pipeline | `analyze_report_v2(text)` — returns `{num_words, entities, tactics}` |
| Heads used | DNRTI-fine-tuned NER (nb 04) + TRAM-fine-tuned tactic classifier (nb 07) |
| Thresholding | Per-tactic tuned thresholds loaded from `thresholds.json` |
| Domain-model A/B | `bert-base-cased` vs `jackaduma/SecBERT` on identical TRAM setup |
| SecBERT saved to | `models/tactic_secbert_final/` |

## Where CTI 201 ends

You now have:

- Two heads fine-tuned on **real** CTI data (DNRTI, TRAM).
- A proper multi-label setup (BCE + sigmoid + per-tactic thresholds).
- Honest evaluation (span-level F1 for NER, macro F1 for multi-label classification).
- A wrapper that ties everything together and a measured comparison against a domain-pretrained model.

What's left for a production system:

- More labeled data (an active-learning loop that prioritizes the most uncertain reports for hand labeling is the classic next step).
- Long-context models (Longformer / ModernBERT) for full multi-page reports, if chunk-and-aggregate hits its limits.
- Proper MLOps — version the models, track evaluations, monitor drift.
- Technique-level (not just tactic-level) classification, once you have enough labels per technique.
- Human-in-the-loop review on the entities the model extracts — especially for actionable ones like IOCs.

Those are outside the scope of a learning series. The 11 notebooks of CTI 101 plus the 8 notebooks of CTI 201 get you to the point where those next steps make sense.